In [1]:
!nvidia-smi

Mon May 18 14:58:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.74                 Driver Version: 591.74         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   32C    P8              3W /   35W |     953MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
import requests

url = r"https://arxiv.org/pdf/2412.19437"
pdf_path = os.path.join(os.getcwd(), "Data_file.pdf")

if not os.path.exists(pdf_path):
    response = requests.get(url)
    if response.status_code == 200:
        with open(pdf_path, "wb") as f:
            f.write(response.content)
        print(f"[INFO] the file is downloaded at : {pdf_path}")
else:
    print(f"[INFO] file exist : {pdf_path} ")


[INFO] file exist : c:\Users\Tarek Hamada Hussien\Downloads\s24153dw\Mohamed diaa test files\Self_traine_RAG_Model\Data_file.pdf 


In [3]:
import fitz  # PyMuPDF

def clean_text(text: str):
    text = text.replace("\n", " ").strip()
    return text

def open_pdf(file_path: str):
    document = fitz.open(file_path)
    pages_and_texts = []
    for page_num, page in enumerate(document):
        text = page.get_text()
        cleaned_text = clean_text(text)
        pages_and_texts.append({
            "text_of_page": cleaned_text,
            "page_count_char": len(cleaned_text),
            "page_count_word": len(cleaned_text.split(" ")),
            "page_count_sentences": len(cleaned_text.split(". ")),
            "tokens_num": len(cleaned_text) // 4,
            "page_number": page_num + 1
        })
    return pages_and_texts

pages_and_texts = open_pdf("Data_file.pdf")
pages_and_texts[3]


{'text_of_page': '1. Introduction In recent years, Large Language Models (LLMs) have been undergoing rapid iteration and evolution (Anthropic, 2024; Google, 2024; OpenAI, 2024a), progressively diminishing the gap to- wards Artificial General Intelligence (AGI). Beyond closed-source models, open-source models, including DeepSeek series (DeepSeek-AI, 2024a,b,c; Guo et al., 2024), LLaMA series (AI@Meta, 2024a,b; Touvron et al., 2023a,b), Qwen series (Qwen, 2023, 2024a,b), and Mistral series (Jiang et al., 2023; Mistral, 2024), are also making significant strides, endeavoring to close the gap with their closed-source counterparts. To further push the boundaries of open-source model capa- bilities, we scale up our models and introduce DeepSeek-V3, a large Mixture-of-Experts (MoE) model with 671B parameters, of which 37B are activated for each token. With a forward-looking perspective, we consistently strive for strong model performance and economical costs. Therefore, in terms of architectu

In [4]:
import pandas as pd
df = pd.DataFrame(pages_and_texts)
df.head(10)


,text_of_page,page_count_char,page_count_word,page_count_sentences,tokens_num,page_number
0,DeepSeek-V3 Technical Report DeepSeek-AI resea...,1817,238,11,454,1
1,Contents 1 Introduction 4 2 Architecture 6 2.1...,2634,919,766,658,2
2,4.5.3 Batch-Wise Load Balance VS. Sequence-Wis...,1715,572,462,428,3
3,"1. Introduction In recent years, Large Languag...",4248,594,26,1062,4
4,Training Costs Pre-Training Context Extension ...,3287,468,21,821,5
5,verification and reflection patterns of R1 int...,3475,457,23,868,6
6,… Router Input Hidden 𝐮𝐮𝑡𝑡 Output Hidden 𝐡𝐡𝑡𝑡 ...,1557,262,8,389,7
7,where c𝐾𝑉 𝑡 ∈R𝑑𝑐is the compressed latent vecto...,2236,353,9,559,8
8,where 𝑁𝑠and 𝑁𝑟denote the numbers of shared exp...,2901,455,17,725,9
9,Main Model (Next Token Prediction) Embedding L...,2904,451,23,726,10


In [5]:
df.describe().round(2)


,page_count_char,page_count_word,page_count_sentences,tokens_num,page_number
count,53.00,53.00,53.00,53.00,53.00
mean,2836.06,474.38,60.85,708.68,27.00
std,799.86,178.18,124.63,199.99,15.44
min,699.00,99.00,1.00,174.00,1.00
25%,2632.00,386.00,17.00,658.00,14.00
50%,2992.00,457.00,23.00,748.00,27.00
75%,3281.00,516.00,29.00,820.00,40.00
max,4248.00,919.00,766.00,1062.00,53.00


# Turn text to sentences 

In [6]:
from spacy.lang.en import English

nlp = English()
nlp.add_pipe("sentencizer")

semtemces = "this my name . hellow mr.ahmed . what is your name ?"
doc = nlp(semtemces)
list(doc.sents)


[this my name ., hellow mr.ahmed ., what is your name ?]

# Now try the `data` 

In [7]:
for page in pages_and_texts:
    page["sentences"] = list(nlp(page["text_of_page"]).sents)
    page["sentences_num"] = len(page["sentences"])

for page in pages_and_texts:
    page["sentences"] = [str(s) for s in page["sentences"]]

pages_and_texts[5]


{'text_of_page': 'verification and reflection patterns of R1 into DeepSeek-V3 and notably improves its reasoning performance. Meanwhile, we also maintain control over the output style and length of DeepSeek-V3. Summary of Core Evaluation Results • Knowledge: (1) On educational benchmarks such as MMLU, MMLU-Pro, and GPQA, DeepSeek-V3 outperforms all other open-source models, achieving 88.5 on MMLU, 75.9 on MMLU-Pro, and 59.1 on GPQA. Its performance is comparable to leading closed-source models like GPT-4o and Claude-Sonnet-3.5, narrowing the gap between open-source and closed-source models in this domain. (2) For factuality benchmarks, DeepSeek-V3 demonstrates superior performance among open-source models on both SimpleQA and Chinese SimpleQA. While it trails behind GPT-4o and Claude-Sonnet-3.5 in English factual knowledge (SimpleQA), it surpasses these models in Chinese factual knowledge (Chinese SimpleQA), highlighting its strength in Chinese factual knowledge. • Code, Math, and Reas

In [8]:
pages_and_texts[5]["sentences"][6]


'Code, Math, and Reasoning: (1) DeepSeek-V3 achieves state-of-the-art performance on math-related benchmarks among all non-long-CoT open-source and closed-source models.'

In [9]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)


,page_count_char,page_count_word,page_count_sentences,tokens_num,page_number,sentences_num
count,53.00,53.00,53.00,53.00,53.00,53.00
mean,2836.06,474.38,60.85,708.68,27.00,23.94
std,799.86,178.18,124.63,199.99,15.44,14.44
min,699.00,99.00,1.00,174.00,1.00,1.00
25%,2632.00,386.00,17.00,658.00,14.00,16.00
50%,2992.00,457.00,23.00,748.00,27.00,23.00
75%,3281.00,516.00,29.00,820.00,40.00,30.00
max,4248.00,919.00,766.00,1062.00,53.00,54.00


In [10]:
test = list(range(1, 25))
slice_size = 10
x = [test[i:i+slice_size] for i in range(0, len(test), slice_size)]
x


[[1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
 [11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
 [21, 22, 23, 24]]

In [11]:
def chunker(input_list: list[str], slice_size: int = slice_size) -> list[list[str]]:
    return [input_list[i:i+slice_size] for i in range(0, len(input_list), slice_size)]

for page in pages_and_texts:
    page["sentences_chunk"] = chunker(page["sentences"])
    page["sentences_chunk_num"] = len(page["sentences_chunk"])

pages_and_texts[5]


{'text_of_page': 'verification and reflection patterns of R1 into DeepSeek-V3 and notably improves its reasoning performance. Meanwhile, we also maintain control over the output style and length of DeepSeek-V3. Summary of Core Evaluation Results • Knowledge: (1) On educational benchmarks such as MMLU, MMLU-Pro, and GPQA, DeepSeek-V3 outperforms all other open-source models, achieving 88.5 on MMLU, 75.9 on MMLU-Pro, and 59.1 on GPQA. Its performance is comparable to leading closed-source models like GPT-4o and Claude-Sonnet-3.5, narrowing the gap between open-source and closed-source models in this domain. (2) For factuality benchmarks, DeepSeek-V3 demonstrates superior performance among open-source models on both SimpleQA and Chinese SimpleQA. While it trails behind GPT-4o and Claude-Sonnet-3.5 in English factual knowledge (SimpleQA), it surpasses these models in Chinese factual knowledge (Chinese SimpleQA), highlighting its strength in Chinese factual knowledge. • Code, Math, and Reas

In [12]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)


,page_count_char,page_count_word,page_count_sentences,tokens_num,page_number,sentences_num,sentences_chunk_num
count,53.00,53.00,53.00,53.00,53.00,53.00,53.00
mean,2836.06,474.38,60.85,708.68,27.00,23.94,2.89
std,799.86,178.18,124.63,199.99,15.44,14.44,1.40
min,699.00,99.00,1.00,174.00,1.00,1.00,1.00
25%,2632.00,386.00,17.00,658.00,14.00,16.00,2.00
50%,2992.00,457.00,23.00,748.00,27.00,23.00,3.00
75%,3281.00,516.00,29.00,820.00,40.00,30.00,3.00
max,4248.00,919.00,766.00,1062.00,53.00,54.00,6.00


In [13]:
pages_and_chunk = []
for page in pages_and_texts:
    for chunk in page["sentences_chunk"]:
        chunk_dict = {}
        paragragh = "".join(chunk).replace("  ", " ").strip()
        chunk_dict["paragraphe"] = paragragh
        chunk_dict["word_count"] = len(paragragh.split(" "))
        chunk_dict["char_count"] = len(paragragh)
        chunk_dict["tokens_count"] = len(paragragh) // 4
        chunk_dict["sentence_count"] = len(chunk)
        chunk_dict["page_number"] = page["page_number"]
        pages_and_chunk.append(chunk_dict)

pages_and_chunk[5], len(pages_and_chunk), len(pages_and_texts)


({'paragraphe': '4.5.3 Batch-Wise Load Balance VS.Sequence-Wise Load Balance . . . . . . . . .27 5 Post-Training 28 5.1 Supervised Fine-Tuning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .28 5.2 Reinforcement Learning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.1 Reward Model . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.2 Group Relative Policy Optimization . . . . . . . . . . . . . . . . . . . . . .30 5.3 Evaluations . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.1 Evaluation Settings . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.2 Standard Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .31 5.3.3 Open-Ended Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . .',
  'word_count': 298,
  'char_count': 830,
  'tokens_count': 207,
  'sentence_count': 10,
  'page_number': 3},
 153,
 53)

In [14]:
df = pd.DataFrame(pages_and_chunk)
df.describe().round(2)


,word_count,char_count,tokens_count,sentence_count,page_number
count,153.00,153.00,153.00,153.00,153.00
mean,157.03,974.48,243.26,8.29,26.52
std,145.88,584.64,146.14,2.95,13.86
min,1.00,1.00,0.00,1.00,1.00
25%,74.00,572.00,143.00,7.00,15.00
50%,134.00,912.00,228.00,10.00,28.00
75%,191.00,1301.00,325.00,10.00,39.00
max,877.00,2975.00,743.00,10.00,53.00


In [15]:
df[df["word_count"] < 4]


,paragraphe,word_count,char_count,tokens_count,sentence_count,page_number
21,9,1,1,0,1,9


In [16]:
filtered_chunks = [chunk for chunk in pages_and_chunk if chunk["tokens_count"] >= 30]
filtered_chunks[111]


{'paragraphe': 'A study of bfloat16 for deep learning training.arXiv preprint arXiv:1905.12322, 2019.S. Krishna, K. Krishna, A. Mohananey, S. Schwarcz, A. Stambler, S. Upadhyay, and M. Faruqui.Fact, fetch, and reason: A unified evaluation of retrieval-augmented generation.CoRR, abs/2409.12941, 2024.doi: 10.48550/ARXIV.2409.12941.URL https://doi.org/10.485 50/arXiv.2409.12941.T. Kwiatkowski, J. Palomaki, O. Redfield, M. Collins, A. P. Parikh, C. Alberti, D. Epstein, I. Polosukhin, J. Devlin, K. Lee, K. Toutanova, L. Jones, M. Kelcey, M. Chang, A. M. Dai, J. Uszkoreit, Q. Le, and S. Petrov.Natural questions: a benchmark for question answering research.Trans.',
 'word_count': 84,
 'char_count': 648,
 'tokens_count': 162,
 'sentence_count': 10,
 'page_number': 39}

In [17]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer("all-MiniLM-L6-v2")


c:\Users\Tarek Hamada Hussien\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6585.37it/s]


In [18]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())


2.6.0+cu124
True


In [19]:
test_set = ["my name", "her name", "the naem"]
embedding = model.encode(test_set)
embedding = dict(zip(test_set, embedding))
embedding, embedding["my name"].shape


({'my name': array([-1.11268580e-01,  3.23448554e-02, -3.83406095e-02,  2.23898105e-02,
         -4.03140672e-02, -5.62008582e-02,  2.33329952e-01,  3.92086990e-02,
         -1.01079093e-02, -6.38170168e-02,  1.64090004e-02, -3.45564187e-02,
          4.02090475e-02, -7.56014735e-02, -8.34890269e-03,  2.65754685e-02,
         -1.42914746e-02,  6.17222628e-03, -8.46135467e-02, -7.72886276e-02,
         -4.17721942e-02,  6.27690554e-02, -3.23730674e-05, -2.02984791e-02,
         -8.92573446e-02,  6.10294519e-03, -3.12835686e-02,  9.11155790e-02,
         -3.69638763e-02, -1.61543325e-01, -3.80753976e-04,  7.48011237e-03,
          9.98783186e-02,  5.72648905e-02, -4.66821576e-03, -4.17357869e-02,
         -1.26731113e-01, -2.11647209e-02,  5.62064126e-02, -1.07810786e-02,
         -4.70564188e-03, -1.19292267e-01,  2.42646392e-02,  5.37947984e-03,
          7.11641647e-03,  7.88039062e-03,  6.28899375e-04,  2.45067067e-02,
          1.61230173e-02,  9.47958305e-02, -9.59324911e-02, -8.88

In [20]:
from tqdm import tqdm

for chunk in tqdm(pages_and_chunk):
    chunk["embedding"] = model.encode(chunk["paragraphe"], device="cuda")


100%|██████████| 153/153 [00:00<00:00, 183.97it/s]


In [21]:
page_and_Embedding_df = pd.DataFrame(pages_and_chunk)
page_and_Embedding_file_name = "page_and_Embedding.csv"
page_and_Embedding_path = os.path.join(os.getcwd(), page_and_Embedding_file_name)
page_and_Embedding_df.to_csv(page_and_Embedding_path, index=False)


In [22]:
pages_and_chunk = pd.read_csv(page_and_Embedding_path)
pages_and_chunk


,paragraphe,word_count,char_count,tokens_count,sentence_count,page_number,embedding
0,DeepSeek-V3 Technical Report DeepSeek-AI resea...,222,1767,441,10,1,[-7.36545101e-02 -5.92405535e-02 5.75507917e-...
1,arXiv:2412.19437v2 [cs.CL] 18 Feb 2025,5,38,9,2,1,[-1.14521928e-01 7.90135004e-03 -7.34651685e-...
2,Contents 1 Introduction 4 2 Architecture 6 2.1...,302,952,238,10,2,[-1.76019724e-02 -5.03365993e-02 -4.91420552e-...
3,14 3.3.1 Mixed Precision Framework . . . . . ....,324,912,228,10,2,[-1.62388142e-02 3.45401727e-02 -1.01859637e-...
4,21 4.2 Hyper-Parameters . . . . . . . . . . . ...,267,742,185,9,2,[ 2.22861432e-02 3.54707316e-02 -1.13855220e-...
...,...,...,...,...,...,...,...
148,1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 1...,877,2961,740,1,49,[-8.76501799e-02 3.37728113e-03 -2.45083477e-...
149,1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 1...,877,2968,742,1,50,[-8.87523293e-02 2.17781449e-03 -2.23974977e-...
150,1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 1...,877,2975,743,1,51,[-8.97151008e-02 9.12356935e-03 -2.74222512e-...
151,1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 1...,877,2975,743,1,52,[-8.86221230e-02 6.90442836e-03 -2.55782288e-...


In [23]:
data = pages_and_chunk.to_dict(orient="records")
data[5]


{'paragraphe': '4.5.3 Batch-Wise Load Balance VS.Sequence-Wise Load Balance . . . . . . . . .27 5 Post-Training 28 5.1 Supervised Fine-Tuning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .28 5.2 Reinforcement Learning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.1 Reward Model . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.2 Group Relative Policy Optimization . . . . . . . . . . . . . . . . . . . . . .30 5.3 Evaluations . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.1 Evaluation Settings . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.2 Standard Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .31 5.3.3 Open-Ended Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . .',
 'word_count': 298,
 'char_count': 830,
 'tokens_count': 207,
 'sentence_count': 10,
 'page_number': 3,
 'embedding': '[-5.76342642e-02  1.86270513e-02

In [24]:
import torch
import numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cuda'

In [25]:
for item in data:
    item['embedding'] = np.fromstring(item['embedding'].strip('[]'), sep=' ')


In [26]:
embeddings = np.array([item["embedding"] for item in data])
embeddings_df = pd.DataFrame(embeddings)
print(embeddings_df.shape)  # should be (153, 384)


(153, 384)


153 rows x 384 dimensions — one embedding vector per chunk.

In [27]:
from time import perf_counter as timer

query = "attentions in models datas"
embeddings_tensor = torch.tensor(np.array(embeddings), dtype=torch.float32)
query_embedding = model.encode(query, device=device)

start = timer()
dot_product = util.dot_score(a=query_embedding, b=embeddings_tensor)[0]
end = timer()
print(f"[INFO]: time it take {(end-start):.5f} seconds")

dot_topk = torch.topk(dot_product, 3)
dot_topk


[INFO]: time it take 0.00043 seconds


torch.return_types.topk(
values=tensor([0.4049, 0.3879, 0.3805]),
indices=tensor([113, 119,  17]))

In [28]:
# just test what idx.item() do
[idx.item() for idx in dot_topk.indices]

#it turn tensor to scaller 

[113, 119, 17]

In [29]:
# Show the top-k matching paragraphs
for score, idx in zip(dot_topk.values, dot_topk.indices):
    print(f"Score: {score:.4f} | Page: {data[idx.item()]['page_number']}")
    print(data[idx.item()]['paragraphe'])
    print("-" * 120)


Score: 0.4049 | Page: 38
In The Thirty-eighth Annual Conference on Neural Information Processing Systems.Y. He, S. Li, J. Liu, Y. Tan, W. Wang, H. Huang, X. Bu, H. Guo, C. Hu, B. Zheng, et al.Chi- nese simpleqa: A chinese factuality evaluation for large language models.arXiv preprint arXiv:2411.07140, 2024.D. Hendrycks, C. Burns, S. Basart, A. Zou, M. Mazeika, D. Song, and J. Steinhardt.Measuring massive multitask language understanding.arXiv preprint arXiv:2009.03300, 2020.38
------------------------------------------------------------------------------------------------------------------------
Score: 0.3879 | Page: 40
Y. Leviathan, M. Kalman, and Y. Matias.Fast inference from transformers via speculative decoding.In International Conference on Machine Learning, ICML 2023, 23-29 July 2023, Honolulu, Hawaii, USA, volume 202 of Proceedings of Machine Learning Research, pages 19274–19286.PMLR, 2023.URL https://proceedings.mlr.press/v202/leviathan23 a.html.H. Li, Y. Zhang, F. Koto, Y. Yan

# functionalize the query search 

In [30]:
def get_answer(query:str,
               embedings_data=embeddings,
               embedding_model: SentenceTransformer=model,
               top_k :int = 5,
               time_taken:bool =True):
    # embed the query
    query_embed= embedding_model.encode(query,convert_to_tensor=True)

    # make sure that data are in the same device (cuda) the GPU in this case
    embedings_data = torch.tensor(np.array(embedings_data), dtype=torch.float32).to('cuda')

    # compare the embedding of query with ather embeddings frome book/dataset
    start = timer()
    dot_product = util.dot_score(a=query_embed, b=embedings_data)[0]
    end = timer()    

    if time_taken:
        print(f"[INFO]time {(end-start):.5f}")
    top_results = torch.topk(dot_product,top_k)
    scores=[score.item() for score in top_results[0]]
    indeces =[score.item() for score in top_results[1]]
    # scores,indeces = top_results
    # print(f"{top_results}")
    return scores,indeces 

def print_top_Result(query:str,
               embedings_data=embeddings,
               embedding_model: SentenceTransformer=model,
               top_k :int = 5,
               time_taken:bool =True,
               pages_and_chunk=data):
    score,indcis=get_answer(query,embedings_data,embedding_model,top_k,time_taken)

    for s, i in zip(score, indcis):
        print(f"Score: {s:.4f} | Index: {i} | Page_number: {pages_and_chunk[i]['page_number']}\nChunk: {pages_and_chunk[i]["paragraphe"]}")
        print("-"*120)

In [31]:
print_top_Result("github")

[INFO]time 0.00014
Score: 0.2818 | Index: 86 | Page_number: 31
Chunk: Standard Evaluation Table 6 presents the evaluation results, showcasing that DeepSeek-V3 stands as the best- performing open-source model.Additionally, it is competitive against frontier closed-source models like GPT-4o and Claude-3.5-Sonnet.31
------------------------------------------------------------------------------------------------------------------------
Score: 0.2415 | Index: 68 | Page_number: 24
Chunk: Math datasets include GSM8K (Cobbe et al.,2021), MATH (Hendrycks et al.,2021), MGSM (Shi et al.,2023), and CMath (Wei et al.,2023).Code datasets include HumanEval (Chen et al.,2021), LiveCodeBench-Base (0801-1101) (Jain et al.,2024), MBPP (Austin et al.,2021), and CRUXEval (Gu et al.,2024).
------------------------------------------------------------------------------------------------------------------------
Score: 0.1890 | Index: 88 | Page_number: 32
Chunk: On the factual knowledge benchmark, SimpleQA, Dee

In [ ]:
# Get GPU available memory
gpu_memory_bytes = torch.cuda.get_device_properties(0).total_memory
gpu_memory_gb = round(gpu_memory_bytes / (2 ** 30))
print(f"Available GPU memory: {gpu_memory_gb} GB")

Available GPU memory: 4 GB
